# ML Ensemble Models for IoT IDS

## Models Implemented:
1. **Random Forest** - Bagging ensemble
2. **XGBoost** - Gradient boosting ensemble
3. **LightGBM** - Gradient boosting ensemble
4. **Voting Classifier** - Hard and soft voting ensemble
5. **Stacking Classifier** - Meta-learning ensemble

## Evaluation Metrics:
- Accuracy, Precision, Recall, F1-Score
- ROC-AUC, PR-AUC
- Confusion Matrix
- Classification Report

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import LabelEncoder
import joblib
import time

# XGBoost and LightGBM
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    print("XGBoost not available. Install with: pip install xgboost")
    XGBOOST_AVAILABLE = False

try:
    import lightgbm as lgb
    LIGHTGBM_AVAILABLE = True
except ImportError:
    print("LightGBM not available. Install with: pip install lightgbm")
    LIGHTGBM_AVAILABLE = False

# Set random seeds for reproducibility
np.random.seed(42)
RANDOM_STATE = 42

# Check GPU availability
def check_gpu_availability():
    """Check if GPU is available for training."""
    gpu_available = False
    try:
        import torch
        gpu_available = torch.cuda.is_available()
    except ImportError:
        pass
    
    # Alternative check using nvidia-ml-py
    if not gpu_available:
        try:
            import pynvml
            pynvml.nvmlInit()
            gpu_available = True
        except:
            pass
    
    return gpu_available

GPU_AVAILABLE = check_gpu_availability()

print("Libraries imported successfully!")
print(f"XGBoost available: {XGBOOST_AVAILABLE}")
print(f"LightGBM available: {LIGHTGBM_AVAILABLE}")
print(f"GPU available: {GPU_AVAILABLE}")

if GPU_AVAILABLE:
    print("GPU acceleration will be enabled for supported models.")
else:
    print("GPU not available. Models will use CPU training.")


## 1. Data Loading and Preprocessing

Load the preprocessed data from the data preprocessing notebook.


In [ ]:
# Load the preprocessed data
def load_preprocessed_data():
    """
    Load and prepare the preprocessed data for ML models.
    This loads the data that was saved from the CSI4999_dataPreprocessing.ipynb notebook.
    """
    try:
        # Load the preprocessed train and test data
        train_data = pd.read_csv('data/preprocessedTrain.csv')
        test_data = pd.read_csv('data/preprocessedTest.csv')
        
        print("Preprocessed data loaded successfully!")
        print(f"Train data shape: {train_data.shape}")
        print(f"Test data shape: {test_data.shape}")
        
        # The last column should be the target (Attack_type or numberTarget)
        print(f"\nColumns in train data: {list(train_data.columns)}")
        
        # Separate features and target
        # Assuming the last column is the target
        X_train = train_data.iloc[:, :-1]  # All columns except the last one
        y_train = train_data.iloc[:, -1]   # Last column is the target
        
        X_test = test_data.iloc[:, :-1]    # All columns except the last one
        y_test = test_data.iloc[:, -1]     # Last column is the target
        
        print(f"\nFeatures shape: {X_train.shape}")
        print(f"Target shape: {y_train.shape}")
        
        return X_train, X_test, y_train, y_test
        
    except FileNotFoundError as e:
        print(f"Error loading preprocessed data: {e}")
        print("Please make sure you have run the CSI4999_dataPreprocessing.ipynb notebook first.")
        print("Expected files: data/preprocessedTrain.csv and data/preprocessedTest.csv")
        return None, None, None, None

# Load the data
X_train, X_test, y_train, y_test = load_preprocessed_data()

if X_train is not None:
    print(f"\n" + "="*50)
    print("DATA LOADED SUCCESSFULLY")
    print("="*50)
    print(f"Training set shape: {X_train.shape}")
    print(f"Test set shape: {X_test.shape}")
    print(f"Training labels shape: {y_train.shape}")
    print(f"Test labels shape: {y_test.shape}")
    print(f"\nClass distribution in training set:")
    print(y_train.value_counts().sort_index())
    print(f"\nClass distribution in test set:")
    print(y_test.value_counts().sort_index())
    
    # Check for any missing values
    print(f"\nMissing values in training set: {X_train.isnull().sum().sum()}")
    print(f"Missing values in test set: {X_test.isnull().sum().sum()}")
    
    # Handle missing values if any exist
    if X_train.isnull().sum().sum() > 0 or X_test.isnull().sum().sum() > 0:
        print("\nHandling missing values...")
        
        # Fill missing values with median for numerical features
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        
        X_train = pd.DataFrame(imputer.fit_transform(X_train), 
                              columns=X_train.columns, 
                              index=X_train.index)
        X_test = pd.DataFrame(imputer.transform(X_test), 
                             columns=X_test.columns, 
                             index=X_test.index)
        
        print(f"Missing values after imputation - Train: {X_train.isnull().sum().sum()}, Test: {X_test.isnull().sum().sum()}")
    
    # Check for infinite values
    print(f"\nInfinite values in training set: {np.isinf(X_train).sum().sum()}")
    print(f"Infinite values in test set: {np.isinf(X_test).sum().sum()}")
    
    # Handle infinite values
    if np.isinf(X_train).sum().sum() > 0 or np.isinf(X_test).sum().sum() > 0:
        print("Handling infinite values...")
        X_train = X_train.replace([np.inf, -np.inf], np.nan)
        X_test = X_test.replace([np.inf, -np.inf], np.nan)
        
        # Fill any remaining NaN values created by inf replacement
        X_train = X_train.fillna(X_train.median())
        X_test = X_test.fillna(X_train.median())  # Use training median for test set
        
        print(f"Infinite values after handling - Train: {np.isinf(X_train).sum().sum()}, Test: {np.isinf(X_test).sum().sum()}")
    
    # Final validation
    print(f"\nFinal data validation:")
    print(f"- Training set shape: {X_train.shape}")
    print(f"- Test set shape: {X_test.shape}")
    print(f"- Missing values: Train={X_train.isnull().sum().sum()}, Test={X_test.isnull().sum().sum()}")
    print(f"- Infinite values: Train={np.isinf(X_train).sum().sum()}, Test={np.isinf(X_test).sum().sum()}")
    
    # Display first few rows
    print(f"\nFirst 5 rows of training features:")
    print(X_train.head())
else:
    print("Failed to load data. Please check the file paths and run the preprocessing notebook first.")


## 2. Model Definition and Configuration

Define all the ensemble models with their hyperparameter configurations.

In [ ]:
# Define model configurations
def get_model_configs():
    """
    Define model configurations for ensemble methods.
    """
    models = {}
    
    # 1. Random Forest
    models['RandomForest'] = {
        'model': RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        'param_grid': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 15, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    }
    
    # 2. XGBoost (if available)
    if XGBOOST_AVAILABLE:
        xgb_params = {
            'n_estimators': 100,
            'max_depth': 6,
            'learning_rate': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': RANDOM_STATE,
            'n_jobs': -1,
            'eval_metric': 'mlogloss'
        }
        
        # Enable GPU if available
        if GPU_AVAILABLE:
            xgb_params['device'] = 'cuda'
            print("XGBoost: GPU acceleration enabled")
        else:
            print("XGBoost: Using CPU training")
            
        models['XGBoost'] = {
            'model': xgb.XGBClassifier(**xgb_params),
            'param_grid': {
                'n_estimators': [50, 100, 200],
                'max_depth': [3, 6, 9],
                'learning_rate': [0.01, 0.1, 0.2],
                'subsample': [0.8, 0.9, 1.0]
            }
        }
    
    # 3. LightGBM (if available)
    if LIGHTGBM_AVAILABLE:
        lgb_params = {
            'n_estimators': 100,
            'max_depth': 6,
            'learning_rate': 0.1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': RANDOM_STATE,
            'n_jobs': -1,
            'class_weight': 'balanced',
            'min_child_samples': 20,  # Increase minimum samples per leaf
            'min_split_gain': 0.0,    # Allow splits with zero gain
            'reg_alpha': 0.1,         # L1 regularization
            'reg_lambda': 0.1,        # L2 regularization
            'verbose': -1             # Suppress warnings
        }
        
        # Enable GPU
        if GPU_AVAILABLE:
            lgb_params['device'] = 'gpu'
            print("LightGBM: GPU acceleration enabled")
        else:
            print("LightGBM: Using CPU training")
            
        models['LightGBM'] = {
            'model': lgb.LGBMClassifier(**lgb_params),
            'param_grid': {
                'n_estimators': [50, 100, 200],
                'max_depth': [3, 6, 9],
                'learning_rate': [0.01, 0.1, 0.2],
                'subsample': [0.8, 0.9, 1.0],
                'min_child_samples': [10, 20, 30],
                'reg_alpha': [0.0, 0.1, 0.2],
                'reg_lambda': [0.0, 0.1, 0.2]
            }
        }
    

    
    return models

# Get model configurations
model_configs = get_model_configs()
print("Available models:")
for name in model_configs.keys():
    print(f"- {name}")


## 3. Ensemble Models Definition

Define voting and stacking ensemble models.


In [ ]:
# Define ensemble models
def get_ensemble_models():
    """
    Define ensemble models (Voting and Stacking).
    """
    ensembles = {}
    
    # Base models for ensemble (using only true ensemble methods)
    base_models = [
        ('rf', RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, class_weight='balanced'))
    ]
    
    if XGBOOST_AVAILABLE:
        xgb_ensemble_params = {'n_estimators': 50, 'random_state': RANDOM_STATE, 'eval_metric': 'mlogloss'}
        if GPU_AVAILABLE:
            xgb_ensemble_params['device'] = 'cuda'
        base_models.append(('xgb', xgb.XGBClassifier(**xgb_ensemble_params)))
    
    if LIGHTGBM_AVAILABLE:
        lgb_ensemble_params = {
            'n_estimators': 50, 
            'random_state': RANDOM_STATE, 
            'class_weight': 'balanced',
            'min_child_samples': 20,
            'min_split_gain': 0.0,
            'reg_alpha': 0.1,
            'reg_lambda': 0.1,
            'verbose': -1
        }
        if GPU_AVAILABLE:
            lgb_ensemble_params['device'] = 'gpu'
        base_models.append(('lgb', lgb.LGBMClassifier(**lgb_ensemble_params)))
    
    # 1. Hard Voting Classifier
    ensembles['HardVoting'] = {
        'model': VotingClassifier(
            estimators=base_models,
            voting='hard'
        ),
        'param_grid': {}
    }
    
    # 2. Soft Voting Classifier
    ensembles['SoftVoting'] = {
        'model': VotingClassifier(
            estimators=base_models,
            voting='soft'
        ),
        'param_grid': {}
    }
    
    # 3. Stacking Classifier
    ensembles['Stacking'] = {
        'model': StackingClassifier(
            estimators=base_models,
            final_estimator=LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced'),
            cv=5
        ),
        'param_grid': {
            'final_estimator__C': [0.1, 1.0, 10.0]
        }
    }
    
    return ensembles

# Get ensemble configurations
ensemble_configs = get_ensemble_models()
print("Available ensemble models:")
for name in ensemble_configs.keys():
    print(f"- {name}")


In [ ]:
# Data validation and cleaning function
def validate_and_clean_data(X_train, X_test, y_train, y_test):
    """
    Comprehensive data validation and cleaning to prevent training errors.
    """
    print("Performing comprehensive data validation and cleaning...")
    
    # Check for missing values
    train_missing = X_train.isnull().sum().sum()
    test_missing = X_test.isnull().sum().sum()
    
    if train_missing > 0 or test_missing > 0:
        print(f"Found missing values - Train: {train_missing}, Test: {test_missing}")
        
        # Use median imputation
        from sklearn.impute import SimpleImputer
        imputer = SimpleImputer(strategy='median')
        
        X_train_clean = pd.DataFrame(
            imputer.fit_transform(X_train), 
            columns=X_train.columns, 
            index=X_train.index
        )
        X_test_clean = pd.DataFrame(
            imputer.transform(X_test), 
            columns=X_test.columns, 
            index=X_test.index
        )
    else:
        X_train_clean = X_train.copy()
        X_test_clean = X_test.copy()
    
    # Check for infinite values
    train_inf = np.isinf(X_train_clean).sum().sum()
    test_inf = np.isinf(X_test_clean).sum().sum()
    
    if train_inf > 0 or test_inf > 0:
        print(f"Found infinite values - Train: {train_inf}, Test: {test_inf}")
        
        # Replace infinite values with NaN, then fill with median
        X_train_clean = X_train_clean.replace([np.inf, -np.inf], np.nan)
        X_test_clean = X_test_clean.replace([np.inf, -np.inf], np.nan)
        
        # Fill remaining NaN values
        X_train_clean = X_train_clean.fillna(X_train_clean.median())
        X_test_clean = X_test_clean.fillna(X_train_clean.median())
    
    # Final validation
    print(f"Final validation results:")
    print(f"- Missing values: Train={X_train_clean.isnull().sum().sum()}, Test={X_test_clean.isnull().sum().sum()}")
    print(f"- Infinite values: Train={np.isinf(X_train_clean).sum().sum()}, Test={np.isinf(X_test_clean).sum().sum()}")
    print(f"- Data types: {X_train_clean.dtypes.value_counts().to_dict()}")
    
    return X_train_clean, X_test_clean, y_train, y_test

print("Data validation function created!")


In [ ]:
# Enhanced training function with early stopping for LightGBM
def train_lightgbm_with_early_stopping(X_train, y_train, X_test, y_test, model_name="LightGBM"):
    """
    Train LightGBM with early stopping to prevent overfitting and warnings.
    """
    print(f"\n{'='*50}")
    print(f"Training {model_name} with Early Stopping")
    print(f"{'='*50}")
    
    start_time = time.time()
    
    # Create train/validation split for early stopping
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
    )
    
    # LightGBM parameters optimized to prevent warnings
    lgb_params = {
        'objective': 'multiclass',
        'num_class': len(np.unique(y_train)),
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.1,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_child_samples': 20,
        'min_split_gain': 0.0,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'verbose': -1,
        'random_state': RANDOM_STATE
    }
    
    # Enable GPU if available
    if GPU_AVAILABLE:
        lgb_params['device'] = 'gpu'
        print("LightGBM: GPU acceleration enabled")
    else:
        print("LightGBM: Using CPU training")
    
    # Create datasets
    train_data = lgb.Dataset(X_train_split, label=y_train_split)
    val_data = lgb.Dataset(X_val_split, label=y_val_split, reference=train_data)
    
    # Train with early stopping
    model = lgb.train(
        lgb_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
    )
    
    # Make predictions
    y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Calculate metrics
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba, model_name)
    
    training_time = time.time() - start_time
    metrics['training_time'] = training_time
    
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Best iteration: {model.best_iteration}")
    
    return model, metrics

print("Enhanced LightGBM training function created!")


## 4. Model Training and Hyperparameter Tuning

Train all models with hyperparameter optimization.


In [ ]:
# Model training and evaluation functions
def train_and_evaluate_model(model, param_grid, X_train, y_train, X_test, y_test, model_name, use_random_search=True):
    """
    Train a model with hyperparameter tuning and evaluate its performance.
    """
    print(f"\n{'='*50}")
    print(f"Training {model_name}")
    print(f"{'='*50}")
    
    start_time = time.time()
    
    # Hyperparameter tuning
    if param_grid:
        if use_random_search:
            search = RandomizedSearchCV(
                model, param_grid, n_iter=20, cv=3, scoring='f1_macro',
                random_state=RANDOM_STATE, n_jobs=-1, verbose=1
            )
        else:
            search = GridSearchCV(
                model, param_grid, cv=3, scoring='f1_macro',
                n_jobs=-1, verbose=1
            )
        
        search.fit(X_train, y_train)
        best_model = search.best_estimator_
        print(f"Best parameters: {search.best_params_}")
        print(f"Best CV score: {search.best_score_:.4f}")
    else:
        best_model = model
        best_model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test) if hasattr(best_model, 'predict_proba') else None
    
    # Calculate metrics
    metrics = calculate_metrics(y_test, y_pred, y_pred_proba, model_name)
    
    training_time = time.time() - start_time
    metrics['training_time'] = training_time
    
    print(f"Training time: {training_time:.2f} seconds")
    
    return best_model, metrics

def calculate_metrics(y_true, y_pred, y_pred_proba, model_name):
    """
    Calculate comprehensive evaluation metrics.
    """
    metrics = {
        'model_name': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro'),
        'recall_macro': recall_score(y_true, y_pred, average='macro'),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
        'precision_weighted': precision_score(y_true, y_pred, average='weighted'),
        'recall_weighted': recall_score(y_true, y_pred, average='weighted'),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted')
    }
    
    # ROC-AUC and PR-AUC (for multiclass)
    if y_pred_proba is not None:
        try:
            metrics['roc_auc_ovr'] = roc_auc_score(y_true, y_pred_proba, multi_class='ovr', average='macro')
            metrics['roc_auc_ovo'] = roc_auc_score(y_true, y_pred_proba, multi_class='ovo', average='macro')
            metrics['pr_auc'] = average_precision_score(y_true, y_pred_proba, average='macro')
        except:
            metrics['roc_auc_ovr'] = None
            metrics['roc_auc_ovo'] = None
            metrics['pr_auc'] = None
    
    return metrics

# Validate and clean data before training
X_train_clean, X_test_clean, y_train_clean, y_test_clean = validate_and_clean_data(X_train, X_test, y_train, y_test)

# Train all individual models
trained_models = {}
all_metrics = []

for model_name, config in model_configs.items():
    try:
        # Use enhanced training for LightGBM
        if model_name == 'LightGBM' and LIGHTGBM_AVAILABLE:
            model, metrics = train_lightgbm_with_early_stopping(
                X_train_clean, y_train_clean, X_test_clean, y_test_clean, model_name
            )
        else:
            model, metrics = train_and_evaluate_model(
                config['model'], config['param_grid'],
                X_train_clean, y_train_clean, X_test_clean, y_test_clean, model_name
            )
        trained_models[model_name] = model
        all_metrics.append(metrics)
    except Exception as e:
        print(f"Error training {model_name}: {str(e)}")
        continue

print("\nIndividual models training completed!")


In [ ]:
# Train ensemble models
print("\nTraining ensemble models...")

for ensemble_name, config in ensemble_configs.items():
    try:
        model, metrics = train_and_evaluate_model(
            config['model'], config['param_grid'],
            X_train_clean, y_train_clean, X_test_clean, y_test_clean, ensemble_name
        )
        trained_models[ensemble_name] = model
        all_metrics.append(metrics)
    except Exception as e:
        print(f"Error training {ensemble_name}: {str(e)}")
        continue

print("\nAll models training completed!")


## 5. Model Performance Comparison

Compare the performance of all trained models.


In [ ]:
# Create performance comparison DataFrame
results_df = pd.DataFrame(all_metrics)
results_df = results_df.set_index('model_name')

# Display results
print("\nModel Performance Comparison:")
print("="*80)

# Sort by F1-macro score
results_sorted = results_df.sort_values('f1_macro', ascending=False)

# Display key metrics
key_metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro', 'training_time']
print(results_sorted[key_metrics].round(4))

# Display ROC-AUC if available
if 'roc_auc_ovr' in results_df.columns:
    print("\nROC-AUC Scores:")
    roc_metrics = ['roc_auc_ovr', 'roc_auc_ovo', 'pr_auc']
    available_roc_metrics = [col for col in roc_metrics if col in results_df.columns]
    print(results_sorted[available_roc_metrics].round(4))


In [ ]:
# Visualize model performance comparison
def plot_model_comparison(results_df):
    """
    Create visualizations for model performance comparison.
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Accuracy comparison
    results_df['accuracy'].plot(kind='bar', ax=axes[0,0], color='skyblue')
    axes[0,0].set_title('Model Accuracy Comparison')
    axes[0,0].set_ylabel('Accuracy')
    axes[0,0].tick_params(axis='x', rotation=45)
    
    # 2. F1-Score comparison
    results_df['f1_macro'].plot(kind='bar', ax=axes[0,1], color='lightgreen')
    axes[0,1].set_title('Model F1-Score (Macro) Comparison')
    axes[0,1].set_ylabel('F1-Score')
    axes[0,1].tick_params(axis='x', rotation=45)
    
    # 3. Precision vs Recall
    axes[1,0].scatter(results_df['precision_macro'], results_df['recall_macro'], 
                     s=100, alpha=0.7, c=range(len(results_df)), cmap='viridis')
    for i, model in enumerate(results_df.index):
        axes[1,0].annotate(model, (results_df['precision_macro'].iloc[i], 
                                  results_df['recall_macro'].iloc[i]),
                          xytext=(5, 5), textcoords='offset points', fontsize=8)
    axes[1,0].set_xlabel('Precision (Macro)')
    axes[1,0].set_ylabel('Recall (Macro)')
    axes[1,0].set_title('Precision vs Recall Trade-off')
    
    # 4. Training time comparison
    results_df['training_time'].plot(kind='bar', ax=axes[1,1], color='orange')
    axes[1,1].set_title('Model Training Time Comparison')
    axes[1,1].set_ylabel('Training Time (seconds)')
    axes[1,1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

# Plot comparison
plot_model_comparison(results_sorted)


## 6. Detailed Model Analysis

Analyze the best performing models in detail.


In [ ]:
# Get the best model
best_model_name = results_sorted.index[0]
best_model = trained_models[best_model_name]

print(f"Best performing model: {best_model_name}")
print(f"F1-Score (Macro): {results_sorted.loc[best_model_name, 'f1_macro']:.4f}")
print(f"Accuracy: {results_sorted.loc[best_model_name, 'accuracy']:.4f}")

# Make predictions with best model
y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test) if hasattr(best_model, 'predict_proba') else None

# Confusion Matrix
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(len(np.unique(y_test))),
            yticklabels=range(len(np.unique(y_test))))
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Classification Report
print(f"\nClassification Report - {best_model_name}:")
print("="*50)
print(classification_report(y_test, y_pred_best))


## 7. Feature Importance Analysis

Analyze feature importance for tree-based models.


In [ ]:
# Feature importance analysis for tree-based models
def plot_feature_importance(model, feature_names, model_name, top_n=20):
    """
    Plot feature importance for tree-based models.
    """
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        indices = np.argsort(importances)[::-1][:top_n]
        
        plt.figure(figsize=(10, 8))
        plt.title(f'Top {top_n} Feature Importances - {model_name}')
        plt.bar(range(top_n), importances[indices])
        plt.xticks(range(top_n), [feature_names[i] for i in indices], rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
        # Return feature importance DataFrame
        feature_importance_df = pd.DataFrame({
            'feature': [feature_names[i] for i in indices],
            'importance': importances[indices]
        })
        return feature_importance_df
    else:
        print(f"{model_name} does not support feature importance analysis.")
        return None

# Plot feature importance for tree-based models
feature_names = X_train.columns.tolist()

for model_name, model in trained_models.items():
    if hasattr(model, 'feature_importances_'):
        print(f"\nFeature Importance Analysis - {model_name}:")
        feature_importance_df = plot_feature_importance(model, feature_names, model_name)
        if feature_importance_df is not None:
            print("\nTop 10 Most Important Features:")
            print(feature_importance_df.head(10))


## 8. Model Saving and Inference Functions

Save the trained models and create inference functions.


In [ ]:
# Save trained models
import os

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

def save_models(trained_models, results_df):
    """
    Save all trained models and results.
    """
    # Save individual models
    for model_name, model in trained_models.items():
        model_path = f'models/{model_name.lower()}_model.pkl'
        joblib.dump(model, model_path)
        print(f"Saved {model_name} to {model_path}")
    
    # Save results
    results_df.to_csv('models/model_performance_results.csv')
    print("Saved model performance results to models/model_performance_results.csv")
    
    # Save best model separately
    best_model_name = results_df.index[0]
    best_model = trained_models[best_model_name]
    joblib.dump(best_model, 'models/best_model.pkl')
    print(f"Saved best model ({best_model_name}) to models/best_model.pkl")

# Save models
save_models(trained_models, results_sorted)


In [ ]:
# Create inference functions
def load_model(model_path):
    """
    Load a saved model.
    """
    return joblib.load(model_path)

def predict_single_sample(model, sample, feature_names=None):
    """
    Make prediction for a single sample.
    """
    if isinstance(sample, dict):
        # Convert dict to DataFrame
        sample_df = pd.DataFrame([sample])
    elif isinstance(sample, (list, np.ndarray)):
        # Convert list/array to DataFrame
        if feature_names is None:
            feature_names = [f'feature_{i}' for i in range(len(sample))]
        sample_df = pd.DataFrame([sample], columns=feature_names)
    else:
        sample_df = sample
    
    prediction = model.predict(sample_df)
    prediction_proba = model.predict_proba(sample_df) if hasattr(model, 'predict_proba') else None
    
    return prediction[0], prediction_proba[0] if prediction_proba is not None else None

def batch_predict(model, X):
    """
    Make predictions for a batch of samples.
    """
    predictions = model.predict(X)
    predictions_proba = model.predict_proba(X) if hasattr(model, 'predict_proba') else None
    
    return predictions, predictions_proba

# Example usage
print("Inference functions created successfully!")
print("\nExample usage:")
print("# Load model")
print("model = load_model('models/best_model.pkl')")
print("\n# Predict single sample")
print("prediction, proba = predict_single_sample(model, sample_data)")
print("\n# Batch predict")
print("predictions, probas = batch_predict(model, X_test)")


## 9. Summary and Conclusions

Summarize the findings and provide recommendations.


In [ ]:
# Final summary
print("\n" + "="*80)
print("ENSEMBLE ML MODELS FOR IOT IDS - FINAL SUMMARY")
print("="*80)

print(f"\nDataset Information:")
print(f"- Training samples: {X_train.shape[0]}")
print(f"- Test samples: {X_test.shape[0]}")
print(f"- Features: {X_train.shape[1]}")
print(f"- Classes: {len(np.unique(y_train))}")

print(f"\nModels Trained: {len(trained_models)}")
for model_name in trained_models.keys():
    print(f"- {model_name}")

print(f"\nTop 3 Performing Models:")
for i, (model_name, row) in enumerate(results_sorted.head(3).iterrows()):
    print(f"{i+1}. {model_name}: F1-Score = {row['f1_macro']:.4f}, Accuracy = {row['accuracy']:.4f}")

print(f"\nBest Model: {best_model_name}")
print(f"- F1-Score (Macro): {results_sorted.loc[best_model_name, 'f1_macro']:.4f}")
print(f"- Accuracy: {results_sorted.loc[best_model_name, 'accuracy']:.4f}")
print(f"- Precision (Macro): {results_sorted.loc[best_model_name, 'precision_macro']:.4f}")
print(f"- Recall (Macro): {results_sorted.loc[best_model_name, 'recall_macro']:.4f}")
print(f"- Training Time: {results_sorted.loc[best_model_name, 'training_time']:.2f} seconds")

print(f"\nRecommendations:")
print(f"1. Use {best_model_name} as the primary model for IoT IDS")
print(f"2. Consider ensemble methods for improved robustness")
print(f"3. Monitor model performance on new data")
print(f"4. Regular retraining with updated data")

print(f"\nFiles Saved:")
print(f"- Individual models: models/*_model.pkl")
print(f"- Best model: models/best_model.pkl")
print(f"- Performance results: models/model_performance_results.csv")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
